<a href="https://colab.research.google.com/github/VI-KA-trs/compling/blob/main/%D0%91%D0%B0%D1%80%D0%B8%D0%BD%D0%BE%D0%B2%D0%B0_fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [2]:
!pip install transformers datasets evaluate accelerate gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [3]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
dataset = load_dataset("ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Датасет загружен. Train: 120000, Test: 7600


In [5]:
dataset['test'][0]

{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.",
 'label': 2}

In [6]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(5000))

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [8]:
training_args = TrainingArguments(
    output_dir="./ag_news_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    #weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    #report_to="tensorboard",
    #logging_steps=500,
)

In [9]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.199635,0.174291,0.943200
2,0.134142,0.192507,0.947600
3,0.089108,0.216689,0.948400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=22500, training_loss=0.1529756590101454, metrics={'train_runtime': 3272.812, 'train_samples_per_second': 109.997, 'train_steps_per_second': 6.875, 'total_flos': 8558812764910464.0, 'train_loss': 0.1529756590101454, 'epoch': 3.0})

In [11]:
eval_results = trainer.evaluate()
print(f"\nEvaluation results: {eval_results}")


Evaluation results: {'eval_loss': 0.2166888415813446, 'eval_accuracy': 0.9484, 'eval_runtime': 13.1504, 'eval_samples_per_second': 380.215, 'eval_steps_per_second': 23.801, 'epoch': 3.0}


In [12]:
model.save_pretrained("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")
print("Модель сохранена в ./ag_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена в ./ag_news_model


In [23]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./ag_news_model", tokenizer="./ag_news_model")

news = [
    "Taylor Swift still can't regain the 'Greatest Pop Star' title from Billboard ... losing out on the accomplishment for the second year in a row ... and begging the question, who outdid her???",
    "Billionaire Telegram founder Pavel Durov revealed he has fathered more than 100 children -- and says they'll all get dibs on his $17 billion fortune!",
    "Russian figure skater Petr Gumennik has been forced to change his short program music two days before the men's program at the Milan Cortina Olympics after joining a growing list of figure skaters dealing with copyright issues."
]

for n in news:
    result = classifier(n)[0]
    print(result)
    print(f"Text: {n}\nCategory: {result['label']}, Score: {result['score']:.4f}\n")



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'label': 'LABEL_0', 'score': 0.9299137592315674}
Text: Taylor Swift still can't regain the 'Greatest Pop Star' title from Billboard ... losing out on the accomplishment for the second year in a row ... and begging the question, who outdid her???
Category: LABEL_0, Score: 0.9299

{'label': 'LABEL_2', 'score': 0.9977793097496033}
Text: Billionaire Telegram founder Pavel Durov revealed he has fathered more than 100 children -- and says they'll all get dibs on his $17 billion fortune!
Category: LABEL_2, Score: 0.9978

{'label': 'LABEL_1', 'score': 0.9979304075241089}
Text: Russian figure skater Petr Gumennik has been forced to change his short program music two days before the men's program at the Milan Cortina Olympics after joining a growing list of figure skaters dealing with copyright issues.
Category: LABEL_1, Score: 0.9979

